<h1 style="color: #CEDDF4;">DEPI Round 4, MS Data Engineer and AI Track</h1>
<h2 style="color: #CEDDF4;" >Final Project: Gold and Oil Prediction System</h2>
<h3 style="color: #CEDDF4;" >Python Code</h3>

<h4 style="color: #CEDDF4;" >1. Import Libraries</h4>

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import requests
import pandas as pd
import time
import csv
from itertools import zip_longest
import math
from pathlib import Path
import os
import yfinance as yf

In [4]:
base_dir = Path.cwd()
raw_data_dir=base_dir/'raw data'
cleaned_data_dir=base_dir/'cleaned data'

<h4 style="color: #CEDDF4;" >2. Egyptian Market Data</h4>
<h5 style="color: #CEDDF4;" >2.1 CPI Rate from IMF</h5>

In [26]:
#this is the API code for getting the CPI (Consumer Products Index) for Egypt from IMF (International Monetary Fund)
#this API is free

#function to be used at any country with any of published indices of IMF

def imf_data(country,index):
    url=f'https://www.imf.org/external/datamapper/api/v1/{index}/{country}'
    response = requests.get(url)
    data=response.json()
    output=data['values'][index][country]
    df = pd.DataFrame(list(output.items()), columns=['year', index])
    return df

#function to be used for IMF data cleaning 
#cl --> cleaned

def cl_imf_data(dataframe,index,country,start_period,end_period):
    dataframe=dataframe[(dataframe['year']>=start_period)&(dataframe['year']<=end_period)]
    dataframe['year']=pd.to_datetime(dataframe['year'])
    dataframe=dataframe.rename(columns={dataframe.columns[1]:'value'})
    dataframe.insert(loc=0, column='id', value=range(1, len(dataframe) + 1)) #to create the key for the values
    dataframe.insert(loc=1,column='country',value=country)
    dataframe.insert(loc=2,column='ticker',value=index)
    return dataframe

In [27]:
#get CPI index data for egypt
egy_cpi_df = imf_data('EGY','PCPIPCH')
egy_cpi_df.to_csv(raw_data_dir/"egypt_data"/"egy_cpi.csv", index=False)

#cleaning of CPI index data for egypt
cl_egy_cpi_df=cl_imf_data(egy_cpi_df,'cpi_imf','egypt','2015','2025')
cl_egy_cpi_df.to_csv(cleaned_data_dir/"egypt_data"/"egy_cpi.csv", index=False)

<h5 style="color: #CEDDF4;" >2.2 Interest Rate from CBE</h5>

In [21]:
#get egyptian data from cbe (central bank of egypt)

#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

#function to add the needed data for cbe statistics
## will not work on exchange rates till now
def cbe_data(target_data,file_name):
    cbe_url=f'https://www.cbe.org.eg/en/economic-research/statistics/{target_data.replace(" ","-")}/historical-data'
    driver.get(cbe_url)
    start_date = "01/01/2016"
    driver.execute_script(f"document.getElementById('fromDate').value = '{start_date}';")
    end_date="31/12/2025"
    driver.execute_script(f"document.getElementById('toDate').value = '{end_date}';")
    driver.find_element(By.XPATH,'/html/body/div[1]/section[3]/div[1]/form/div[1]/div/button/div/span[2]').click()
    time.sleep(1)
    driver.find_element(By.XPATH,'//*[@id="historicalDataForm"]/div[2]/button[1]/span').click()
    downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
    downloaded_filename.rename(raw_data_dir /"egypt_data"/ f"{file_name}.csv")
    return file_name

In [22]:
cbe_inflation=cbe_data('inflation rates','egy_cbe_inflation_rate')

In [41]:
def cl_cbe_data(df,country):
    df = pd.read_excel(raw_data_dir  /"egypt_data"/ 'egy_cbe_inflation_rate.csv')
    df=df.drop(columns=['Unnamed: 3','Unnamed: 4'])
    df=df.rename(columns={df.columns[0]:'date',df.columns[1]:'headline_inflation_yy',df.columns[2]:'core_inflation_yy'})
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%d/%m/%Y')
    df = df.dropna(subset=['date'])
    df = df.reset_index(drop=True)
    #to transpose the values of the multiple tickers
    column_values=[col for col in df.columns if col != 'date']
    df=df.melt(id_vars=['date'],value_vars=column_values,var_name='ticker',value_name='value')
    df['value'] = df['value'].str.replace('%', '').astype(float)
    df.insert(loc=1,column='country',value=country)
    df = df.sort_values('date').reset_index(drop=True)
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    return df

In [42]:
cl_cbe_data_df=cl_cbe_data(cbe_inflation,'egypt')
cl_cbe_data_df.to_csv(cleaned_data_dir/"egypt_data"/"egy_cbe_inflation_rates.csv", index=False)

C:\Users\HMSY\AppData\Local\Temp\ipykernel_22808\3029369841.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%d/%m/%Y')


<h5 style="color: #CEDDF4;" >2.3 USD/EGP Exchange Rate from CBE</h5>

In [34]:
#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

#get exchange rate between usd and egp from cbe
def cbe_usdegp_er(start_date,end_date,file_name):
    #file_name='egy_usdegp_rate'
    er_cbe_url='https://www.cbe.org.eg/en/markets/foreign-exchange/foreign-exchange-historical-data'
    driver.get(er_cbe_url)
    time.sleep(5)
    #start_date = "01/01/2016"
    driver.execute_script(f"document.getElementById('fromDate').value = '{start_date}';")
    #end_date="31/12/2025"
    driver.execute_script(f"document.getElementById('toDate').value = '{end_date}';")
    driver.find_element(By.XPATH,'/html/body/div[1]/section[3]/div[1]/form/div[1]/div/button/div/span[2]/span').click()
    time.sleep(1)
    driver.find_element(By.XPATH,'//*[@id="historicalDataForm"]/div[2]/button[1]').click()
    time.sleep(2)
    downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
    time.sleep(2)
    downloaded_filename.rename(raw_data_dir / "egypt_data" / f"{file_name}{downloaded_filename.suffix}")
    return file_name


In [35]:
cbe_usdegp_er('01/01/2016','31/12/2025','egy_cbe_usdegp_rate')

'egy_cbe_usdegp_rate'

In [53]:
def cl_cbe_usdegp(df, country, ticker):
    df = pd.read_excel(raw_data_dir / "egypt_data" / 'egy_cbe_usdegp_rate.xlsx')
    df = df.rename(columns={df.columns[0]: 'date', df.columns[1]: 'value'})
    df.insert(loc=0, column='country', value=country)
    df.insert(loc=1, column='ticker', value=ticker)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')  # force all to datetime
    df = df.dropna(subset=['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df['date'] = df['date'].dt.strftime('%d/%m/%Y')
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    return df

In [55]:
cl_cbe_usdegp_rate = cl_cbe_usdegp('usdegp_rate','egypt','usdegp')
cl_cbe_usdegp_rate.to_csv(cleaned_data_dir/"egypt_data"/"egy_usdegp_rate.csv", index=False)

C:\Users\HMSY\AppData\Local\Temp\ipykernel_22808\1174431052.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')  # force all to datetime


<h5 style="color: #CEDDF4;" >2.4 Subsidies on Electrical / Petroleum Products in Egyptian Budget from MOF</h5>

In [ ]:
def cl_egy_budget(filename1,filename2,cleaned_filename):
    df1 = pd.read_excel(raw_data_dir / "egypt_data" / f'{filename1}.xlsx')
    keyword1 = filename1.split('_on_')[1].split('_expected')[0]
    df1 = df1.rename(columns={df1.columns[0]: 'date', df1.columns[1]: f"planned_subsidy_{keyword1}",df1.columns[1]: f'actual_subsidy_{keyword1}'})
    #df.insert(loc=0, column='country', value='egypt')
    #to transpose the values of the multiple tickers
    column_values=[col for col in df1.columns if col != 'date']
    df1=df1.melt(id_vars=['date'],value_vars=column_values,var_name='ticker',value_name='value_in_millions')
    #df['value'] = df['value'].str.replace('%', '').astype(float)
    df1.insert(loc=0,column='country',value='egypt')
    df1 = df1.sort_values('date').reset_index(drop=True)
    df1.insert(loc=0, column='id', value=range(1, len(df1) + 1))
    df1.to_csv(cleaned_data_dir/"egypt_data"/f"{cleaned_filename}.csv", index=False)

    df2 = pd.read_excel(raw_data_dir / "egypt_data" / f'{filename2}.xlsx')
    keyword2 = filename2.split('_on_')[1].split('_expected')[0]
    df2 = df2.rename(columns={df2.columns[0]: 'date', df2.columns[1]: f"planned_subsidy_{keyword2}",df2.columns[1]: f'actual_subsidy_{keyword2}'})
    #df.insert(loc=0, column='country', value='egypt')
    #to transpose the values of the multiple tickers
    column_values=[col for col in df2.columns if col != 'date']
    df2=df2.melt(id_vars=['date'],value_vars=column_values,var_name='ticker',value_name='value_in_millions')
    #df['value'] = df['value'].str.replace('%', '').astype(float)
    df2.insert(loc=0,column='country',value='egypt')
    df2 = df2.sort_values('date').reset_index(drop=True)
    df2.insert(loc=0, column='id', value=range(1, len(df2) + 1))
    df2.to_csv(cleaned_data_dir/"egypt_data"/f"{cleaned_filename}.csv", index=False)
    return df1,df2

In [72]:
def cl_egy_budget(filename1,filename2,cleaned_filename):
    df1 = pd.read_excel(raw_data_dir / "egypt_data" / f'{filename1}.xlsx')
    keyword1 = filename1.split('_on_')[1].split('_expected')[0]
    df1 = df1.rename(columns={df1.columns[0]: 'date', df1.columns[1]: f"planned_subsidy_{keyword1}",df1.columns[2]: f'actual_subsidy_{keyword1}'})
    df1[f'actual_subsidy_{keyword1}'] = df1[f'actual_subsidy_{keyword1}'].fillna(df1[f'planned_subsidy_{keyword1}'])

    df2 = pd.read_excel(raw_data_dir / "egypt_data" / f'{filename2}.xlsx')
    keyword2 = filename2.split('_on_')[1].split('_expected')[0]
    df2 = df2.rename(columns={df2.columns[0]: 'date', df2.columns[1]: f"planned_subsidy_{keyword2}",df2.columns[2]: f'actual_subsidy_{keyword2}'})
    df2[f'actual_subsidy_{keyword2}'] = df2[f'actual_subsidy_{keyword2}'].fillna(df2[f'planned_subsidy_{keyword2}'])

    df=pd.merge(df1,df2,on='date',how='inner')
    #df.insert(loc=0, column='country', value='egypt')
    #to transpose the values of the multiple tickers
    column_values=[col for col in df.columns if col != 'date']
    df=df.melt(id_vars=['date'],value_vars=column_values,var_name='ticker',value_name='value_in_millions')
    #df['value'] = df['value'].str.replace('%', '').astype(float)
    df.insert(loc=0,column='country',value='egypt')
    df = df.sort_values('date').reset_index(drop=True)
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    df.to_csv(cleaned_data_dir/"egypt_data"/f"{cleaned_filename}.csv", index=False)
    return

In [73]:
cl_egy_budget('subsidies_on_electricity_expected_vs_actual','subsidies_on_petroleum_products_expected_vs_actual','egy_subsidy_in_budget')